# Algorithms in ySights

This tutorial covers the analytical algorithms available in ySights for understanding simulation dynamics.

## What You'll Learn

- Profile similarity analysis
- Visibility paradox detection
- Recommendation system metrics
- Topic dynamics analysis

---

In [ ]:
from ysights import YDataHandler
from ysights.algorithms import (
    profile_topics_similarity,
    visibility_paradox,
    user_visibility_vs_neighbors,
    visibility_paradox_population_size_null,
    engagement_momentum,
    personalization_balance_score,
    topic_spread,
    adoption_rate,
    peak_engagement_time
)
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Initialize data handler and get network
db_path = 'path/to/your/simulation.db'
ydh = YDataHandler(db_path)
network = ydh.social_network()

## 1. Profile Similarity Analysis

Analyzes how similar users' interest profiles are across the network.

In [ ]:
# Calculate profile similarity
# This computes cosine similarity between users' topic interest profiles
similarity_scores = profile_topics_similarity(ydh, network)

print(f"Computed {len(similarity_scores)} similarity scores")
print(f"\nSimilarity Statistics:")
print(f"  Mean: {np.mean(similarity_scores):.4f}")
print(f"  Median: {np.median(similarity_scores):.4f}")
print(f"  Std Dev: {np.std(similarity_scores):.4f}")
print(f"  Min: {min(similarity_scores):.4f}")
print(f"  Max: {max(similarity_scores):.4f}")

In [ ]:
# Visualize similarity distribution
plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
plt.hist(similarity_scores, bins=50, edgecolor='black', alpha=0.7, color='coral')
plt.xlabel('Similarity Score', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.title('Profile Similarity Distribution', fontsize=13, fontweight='bold')
plt.axvline(np.mean(similarity_scores), color='red', linestyle='--', 
            label=f'Mean: {np.mean(similarity_scores):.3f}')
plt.legend()
plt.grid(True, alpha=0.3)

# Box plot
plt.subplot(1, 2, 2)
plt.boxplot(similarity_scores, vert=True)
plt.ylabel('Similarity Score', fontsize=11)
plt.title('Profile Similarity Box Plot', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Visibility Paradox Analysis

The visibility paradox occurs when users' posts receive less visibility than their friends' posts on average.

In [ ]:
# Analyze visibility paradox
# N is the sample size (use smaller number for faster computation)
paradox_results = visibility_paradox(ydh, network, N=100)

print("Visibility Paradox Results:")
print(f"  Paradox Score: {paradox_results['paradox_score']:.4f}")
print(f"  Affected Users: {paradox_results['affected_count']} out of {paradox_results['total_users']}")
print(f"  Percentage: {paradox_results['affected_percentage']:.2f}%")

### User Visibility vs. Neighbors' Visibility

In [ ]:
# Get detailed visibility comparison
user_vis, neighbor_vis = user_visibility_vs_neighbors(ydh, network)

print(f"Analyzed {len(user_vis)} users")
print(f"\nAverage User Visibility: {np.mean(user_vis):.2f}")
print(f"Average Neighbor Visibility: {np.mean(neighbor_vis):.2f}")

# Count users experiencing paradox
paradox_count = sum(1 for u, n in zip(user_vis, neighbor_vis) if u < n)
print(f"\nUsers experiencing paradox: {paradox_count} ({100*paradox_count/len(user_vis):.1f}%)")

In [ ]:
# Visualize user vs neighbor visibility
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(user_vis, neighbor_vis, alpha=0.5, s=30)
plt.xlabel('User Visibility', fontsize=12)
plt.ylabel('Neighbor Average Visibility', fontsize=12)
plt.title('Visibility Paradox: User vs. Neighbors', fontsize=14, fontweight='bold')

# Add diagonal line (equality)
max_val = max(max(user_vis), max(neighbor_vis))
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Equality Line')

# Shade paradox region
plt.fill_between([0, max_val], [0, max_val], max_val, alpha=0.1, color='red', 
                 label='Paradox Region (neighbors more visible)')

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Recommendation System Metrics

### Engagement Momentum

Measures how consistently users engage with recommended content over time.

In [ ]:
# Calculate engagement momentum
# time_window_rounds specifies how many rounds to analyze
momentum = engagement_momentum(ydh, time_window_rounds=24)

print("Engagement Momentum Analysis:")
print(f"  Users analyzed: {len(momentum)}")
print(f"  Average momentum: {np.mean(list(momentum.values())):.4f}")
print(f"  Median momentum: {np.median(list(momentum.values())):.4f}")

# Show top 5 users by momentum
top_momentum = sorted(momentum.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 Users by Engagement Momentum:")
for i, (user, score) in enumerate(top_momentum, 1):
    print(f"  {i}. User {user}: {score:.4f}")

### Personalization Balance Score

Measures how well the recommendation system balances exploration vs. exploitation.

In [ ]:
# Calculate personalization balance
balance_scores = personalization_balance_score(ydh)

print("Personalization Balance Analysis:")
print(f"  Users analyzed: {len(balance_scores)}")
print(f"  Average balance: {np.mean(list(balance_scores.values())):.4f}")
print(f"  Median balance: {np.median(list(balance_scores.values())):.4f}")
print("\nInterpretation:")
print("  Score → 0: Heavy exploitation (narrow recommendations)")
print("  Score → 1: Heavy exploration (diverse recommendations)")

In [ ]:
# Visualize balance distribution
plt.figure(figsize=(10, 6))
plt.hist(list(balance_scores.values()), bins=30, edgecolor='black', alpha=0.7, color='seagreen')
plt.xlabel('Personalization Balance Score', fontsize=12)
plt.ylabel('Number of Users', fontsize=12)
plt.title('Distribution of Personalization Balance', fontsize=14, fontweight='bold')
plt.axvline(0.5, color='red', linestyle='--', linewidth=2, label='Perfect Balance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Topic Dynamics Analysis

### Topic Spread

Analyzes how a topic spreads across the network over time.

In [ ]:
# Analyze spread of a specific topic
topic_id = 5  # Change to any topic ID in your simulation
spread = topic_spread(ydh, topic_id)

print(f"Topic {topic_id} Spread Analysis:")
print(f"  Total unique users: {spread['unique_users']}")
print(f"  Total posts: {spread['total_posts']}")
print(f"  Spread rate: {spread['spread_rate']:.4f} users/post")
print(f"  Active rounds: {spread['rounds_active']}")

### Adoption Rate

Measures how quickly users adopt a topic.

In [ ]:
# Calculate adoption rate
adoption = adoption_rate(ydh, topic_id)

print(f"Topic {topic_id} Adoption Rate:")
print(f"  Rate: {adoption['adoption_rate']:.4f} users/round")
print(f"  Total adopters: {adoption['total_adopters']}")
print(f"  Time span: {adoption['time_span']} rounds")

### Peak Engagement Time

Identifies when a topic receives maximum engagement.

In [ ]:
# Find peak engagement time
peak = peak_engagement_time(ydh, topic_id)

print(f"Topic {topic_id} Peak Engagement:")
print(f"  Peak round: {peak['peak_round']}")
print(f"  Peak engagement: {peak['peak_engagement']} interactions")
print(f"  Average engagement: {peak['avg_engagement']:.2f} interactions/round")

### Comparing Multiple Topics

In [ ]:
# Compare adoption rates for multiple topics
topics_to_compare = [1, 2, 3, 4, 5]  # Change to your topic IDs

topic_metrics = {}
for topic_id in topics_to_compare:
    try:
        adoption = adoption_rate(ydh, topic_id)
        peak = peak_engagement_time(ydh, topic_id)
        topic_metrics[topic_id] = {
            'adoption_rate': adoption['adoption_rate'],
            'peak_engagement': peak['peak_engagement']
        }
    except:
        pass  # Skip topics with no data

# Visualize comparison
if topic_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    topics = list(topic_metrics.keys())
    adoption_rates = [topic_metrics[t]['adoption_rate'] for t in topics]
    peak_engagements = [topic_metrics[t]['peak_engagement'] for t in topics]
    
    # Adoption rates
    axes[0].bar([f'Topic {t}' for t in topics], adoption_rates, color='skyblue', edgecolor='black')
    axes[0].set_ylabel('Adoption Rate (users/round)', fontsize=11)
    axes[0].set_title('Topic Adoption Rates', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Peak engagements
    axes[1].bar([f'Topic {t}' for t in topics], peak_engagements, color='lightcoral', edgecolor='black')
    axes[1].set_ylabel('Peak Engagement', fontsize=11)
    axes[1].set_title('Peak Topic Engagement', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## Summary

In this tutorial, you learned:

✓ How to measure profile similarity across the network  
✓ Detecting and analyzing the visibility paradox  
✓ Computing recommendation system metrics (engagement momentum, personalization balance)  
✓ Analyzing topic dynamics (spread, adoption rate, peak engagement)  
✓ Visualizing algorithmic results for interpretation  
✓ Comparing metrics across multiple topics  

## Next Steps

- **Visualization Tutorial**: Create publication-ready visualizations using ySights' viz module